# 03 — Australia Murray-Darling Basin, HLS with Prithvi Engine

Compares two Prithvi-based delineation modes on the Murray-Darling Basin using **Harmonized Landsat-Sentinel (HLS)** imagery:

1. **Prithvi ViT embedding** — full encoder feature extraction + clustering
2. **PCA baseline** — spectral PCA + clustering (no model, CPU only)

Both run on the same AOI for 2022–2024.

**Estimated runtime:** ~45–90 minutes (3 years, GPU recommended for ViT mode).

**Prerequisites:**
```bash
pip install agribound[gee,prithvi]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/australia_murray_darling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2022, 2025)
SOURCE = "hls"
ENGINE = "prithvi"

## Create Study Area

An AOI in the Murray-Darling Basin covering large irrigated agriculture.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [149.70, -30.30],
                        [149.85, -30.30],
                        [149.85, -30.15],
                        [149.70, -30.15],
                        [149.70, -30.30],
                    ]
                ],
            },
            "properties": {"name": "Murray-Darling Basin AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "murray_darling_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation: Prithvi ViT vs PCA (2022–2024)

In [ ]:
comparison_results = {}

for year in YEARS:
    print(f"\n{'=' * 50}\nYear {year}\n{'=' * 50}")

    # Prithvi ViT embedding mode
    print("  Mode 1: Prithvi ViT embeddings")
    vit_path = OUTPUT_DIR / f"fields_hls_vit_{year}.gpkg"
    gdf_vit = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=year,
        engine=ENGINE,
        output_path=str(vit_path),
        gee_project=GEE_PROJECT,
        composite_method="median",
        min_area=5000,
        simplify=3.0,
        engine_params={"mode": "embed"},
    )
    comparison_results[f"ViT {year}"] = gdf_vit
    print(f"    ViT: {len(gdf_vit)} fields")

    # PCA baseline mode
    print("  Mode 2: PCA baseline")
    pca_path = OUTPUT_DIR / f"fields_hls_pca_{year}.gpkg"
    gdf_pca = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=year,
        engine=ENGINE,
        output_path=str(pca_path),
        gee_project=GEE_PROJECT,
        composite_method="median",
        min_area=5000,
        simplify=3.0,
        engine_params={"mode": "pca"},
    )
    comparison_results[f"PCA {year}"] = gdf_pca
    print(f"    PCA: {len(gdf_pca)} fields")

# Summary
print(f"\n{'Method':<25} {'Fields':>8} {'Area (ha)':>12}")
print(f"{'-' * 25} {'-' * 8} {'-' * 12}")
for label, gdf in comparison_results.items():
    area = gdf["metrics:area"].sum() / 10000 if "metrics:area" in gdf.columns else 0
    print(f"{label:<25} {len(gdf):>8} {area:>12,.1f}")

## Visualization

In [ ]:
if comparison_results:
    from agribound.visualize import show_comparison

    m = show_comparison(
        list(comparison_results.values()),
        labels=list(comparison_results.keys()),
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_murray_darling.html"),
    )
    m